# PFAS Groundwater — HGT / R-GCN Multi-Relational T1 (Colab)

**Task T1a**: predict EPA-2024 regulatory PFAS exceedance (binary) from context features only  
(strict predictive mode — no PFAS measurement as a feature).

**This notebook is AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the dataset via
`git clone` (no Google Drive), installs PyTorch Geometric for the Colab torch wheel, builds
a **homogeneous well-well multi-relational graph** (two edge types), trains HGT and R-GCN,
and reports the honest triplet **(spatial AUC, random AUC, Δ)** vs the non-graph WALL and
gnn_phase1 single-relation baselines.

**Graph design** (eval-validated, `experiments/hgt_rgcn_t1/eval_validation.md`):
- Node type: **well** only (no fabricated source/installation nodes — they don't exist, C-NODE.1/2)
- Relation 1 `('well','near','well')`: spatial k-NN, hard cap 1.5 km (C-SPAT.3)
- Relation 2 `('well','same_subbasin_knn','well')`: k-NN restricted to same SGMA sub-basin, cap 2 km
- HGT / R-GCN = **multi-relational encoders** over a single real node type (C-NODE.3)
- Cross-block edges cut **per relation** + asserted to 0 (C-SPAT.2/5)
- Inductive: message-passing uses TRAIN-TRAIN edges only during training (C-SPAT.4)
- lat/lon NOT node features; geography enters only through topology (C-LOC.1)
- Evaluation at the sampling (row) level for strict comparability with the non-graph WALL

**Spatial-leakage reference** (gnn_phase1): GraphSAGE Δ≈+0.20, GCN Δ≈+0.22 → random split
inflates AUC by ~0.20 pt on this dataset; honest result is the spatial AUC.

In [ ]:
# ============================================================
#  USER PARAMETERS — edit here, then Runtime > Run all
# ============================================================
SMOKE_TEST = False        # True = fast CPU sanity check (~3 min); False = full GPU run

REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"        # commit SHA or branch to clone
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"
EXP_DIR  = "experiments/hgt_rgcn_t1"

# Models to compare
MODELS    = ["hgt", "rgcn"]   # also supports 'hetero_sage'

# Graph topology
K_SPATIAL      = 8
CAP_KM_SPATIAL = 1.5    # hard great-circle cap for the spatial relation (km)
K_SUBBASIN     = 8
CAP_KM_SUBBASIN = 2.0   # hard cap for the intra-sub-basin relation (km)

# CV protocol (must match gnn_phase1 for valid comparison)
N_BLOCKS = 8             # spatial KMeans blocks (primary reference) = random folds (Δ)
COMPUTE_DELTA = True     # also run random CV to measure Δ(random − spatial)

# Model architecture
HIDDEN     = 128
LAYERS     = 3
DROPOUT    = 0.3
HEADS      = 4           # HGT attention heads (ignored by R-GCN)

# Training
MAX_EPOCHS = 400
PATIENCE   = 50
LR         = 5e-3
WEIGHT_DECAY = 5e-4
SEED       = 42

## Cell 1 — GPU detection & versions

In [ ]:
import sys, platform, subprocess
print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")
try:
    import torch
    print(f"torch   : {torch.__version__}  CUDA avail: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        print(f"CUDA    : {torch.version.cuda}")
except ImportError:
    print("torch not yet installed")

## Cell 2 — Clone repo (code + versioned dataset, no Drive)

In [ ]:
import os
REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
assert os.path.exists(DATA_PATH), (
    f"FATAL: dataset missing at {DATA_PATH}. "
    "The clone should have brought data/ — check REPO_URL and GIT_REF."
)
print("workdir:", os.getcwd())
print("dataset:", DATA_PATH, "—", os.path.getsize(DATA_PATH) // 1024, "KB")

## Cell 3 — Install PyTorch Geometric for the runtime's torch wheel

PyG compiled extensions must match `torch.__version__` + CUDA tag. We detect both at
runtime and install from the matching wheel index (idempotent if already importable).

In [ ]:
def ensure_pyg():
    try:
        import torch_geometric
        print("torch_geometric already present:", torch_geometric.__version__)
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print("Installing PyG extensions from", idx)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "-f", idx], check=True)
    # Optional compiled helpers (speed up conv; ignore failure on CPU)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse", "-f", idx])
    import torch_geometric
    print("Installed torch_geometric:", torch_geometric.__version__)

ensure_pyg()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=14.0", "pyyaml>=6.0", "scikit-learn>=1.4"])

# Verify import works after install
import torch_geometric  # noqa
from torch_geometric.nn import HGTConv, RGCNConv  # noqa
print("PyG import OK — HGTConv and RGCNConv available")

## Cell 4 — Load dataset & integrity check

In [ ]:
import pandas as pd
from src import config as C
from src import targets as T

df = pd.read_parquet(DATA_PATH)
n_rows, n_cols = df.shape
n_wells = df[C.WELL_ID].nunique()
print(f"rows={n_rows}  cols={n_cols}  wells={n_wells}")
if not SMOKE_TEST:
    assert n_rows == 46338, f"Unexpected row count {n_rows} (expected 46338)"
    assert n_wells == 11333, f"Unexpected well count {n_wells} (expected 11333)"
    for col in [C.WELL_ID, C.LAT, C.LON, C.SGMA_SUBBASIN]:
        assert col in df.columns, f"FATAL: required column '{col}' missing from dataset"
print(f"T1a prevalence: {float(T.build_T1a(df).mean()):.4f}")
print("Dataset integrity OK")

## Cell 5 — Smoke-test (SMOKE_TEST=True only)

Subsample ~400 wells, 3 folds, 15 epochs — end-to-end CPU check in ~3 min.  
Validates graph construction, model forward pass, metric computation, and the  
anti-leakage assertions (cross_block_total must be 0 for each model).

In [ ]:
from src import gnn_hetero_t1 as H
import time

if SMOKE_TEST:
    smoke_results = {}
    for model_name in MODELS:
        print(f"\n=== SMOKE-TEST {model_name.upper()} ===")
        t0 = time.time()
        res = H.run_t1_multirel_cv(
            df, name=model_name, regime="spatial", smoke=True,
            compute_delta=True, write=True,
            k_spatial=K_SPATIAL, cap_km_spatial=CAP_KM_SPATIAL,
            k_subbasin=K_SUBBASIN, cap_km_subbasin=CAP_KM_SUBBASIN,
            hidden=min(HIDDEN, 64), layers=min(LAYERS, 2), dropout=DROPOUT, heads=HEADS,
            max_epochs=15, patience=6, seed=SEED,
            exp_dir=EXP_DIR, verbose=True)
        elapsed = time.time() - t0
        sp = res["spatial"]
        assert sp["n_cross_block_total"] == 0, (
            f"LEAK DETECTED: {sp['n_cross_block_total']} cross-block edges remain!")
        smoke_results[model_name] = sp["auc_oof_global"]
        print(f"{model_name}: spatial AUC={sp['auc_oof_global']:.4f}  "
              f"cross_block=0 ✓  elapsed={elapsed:.0f}s")
        # Rough extrapolation to full run
        scale = (11333 / 400) * (MAX_EPOCHS / 15) * (N_BLOCKS / 3)
        print(f"  ≈ estimated full-run time: {elapsed * scale / 60:.0f} min")

    print("\n=== SMOKE-TEST PASSED for all models ===")
    for m, auc in smoke_results.items():
        print(f"  {m}: AUC={auc:.4f} (smoke subset, not representative)")
else:
    print("SMOKE_TEST=False — skip to Cell 6 for the full GPU run.")

## Cells 6–7 — Full runs (SMOKE_TEST=False)

Each model runs spatial CV (8 blocks) + optionally random CV (Δ measurement).  
Results stream per fold with verbose=True. Checkpointing: metrics written incrementally  
to `experiments/hgt_rgcn_t1/` after each model completes.

In [ ]:
# Cell 6 — Full run: HGT
if not SMOKE_TEST:
    import time, json
    from src import gnn_hetero_t1 as H

    print("=" * 60)
    print("HGT  —  multi-relational T1a  —  spatial CV (8 blocs)")
    print("=" * 60)
    t0 = time.time()
    res_hgt = H.run_t1_multirel_cv(
        df, name="hgt", regime="spatial",
        compute_delta=COMPUTE_DELTA, write=True,
        k_spatial=K_SPATIAL, cap_km_spatial=CAP_KM_SPATIAL,
        k_subbasin=K_SUBBASIN, cap_km_subbasin=CAP_KM_SUBBASIN,
        hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT, heads=HEADS,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, lr=LR,
        weight_decay=WEIGHT_DECAY, n_blocks=N_BLOCKS, seed=SEED,
        exp_dir=EXP_DIR, verbose=True)
    elapsed_hgt = time.time() - t0

    sp = res_hgt["spatial"]
    gm = sp["global_metrics"]
    ci = sp["auc_oof_ci95"]
    assert sp["n_cross_block_total"] == 0, "LEAK: cross-block edges remain!"

    print(f"\nHGT spatial AUC : {sp['auc_oof_global']:.4f}  "
          f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]")
    print(f"        mean±std: {sp['auc_mean']:.4f} ± {sp['auc_std']:.4f}")
    print(f"        F1@OOF  : {gm['f1']:.4f}   PR-AUC: {gm['pr_auc']:.4f}")
    print(f"        Brier   : {gm['brier']:.4f}   ECE: {gm.get('ece', float('nan')):.4f}")
    if COMPUTE_DELTA and "delta_random_minus_spatial" in res_hgt:
        print(f"        Δ(rand−spatial): {res_hgt['delta_random_minus_spatial']:+.4f}")
    print(f"        cross_block_total: {sp['n_cross_block_total']}  (must be 0)")
    print(f"Elapsed: {elapsed_hgt/60:.1f} min")

    # Incremental checkpoint
    import os; os.makedirs(EXP_DIR, exist_ok=True)
    with open(f"{EXP_DIR}/metrics_hgt.json", "w") as f:
        def _cvt(o):
            import numpy as np
            if isinstance(o, (np.floating,)): return float(o)
            if isinstance(o, (np.integer,)): return int(o)
            if isinstance(o, np.ndarray): return o.tolist()
            return str(o)
        json.dump(res_hgt, f, indent=2, default=_cvt)
    print("Checkpoint written to", f"{EXP_DIR}/metrics_hgt.json")
else:
    res_hgt = None
    print("SMOKE_TEST=True — skipping full HGT run.")

In [ ]:
# Cell 7 — Full run: R-GCN
if not SMOKE_TEST:
    import time

    print("=" * 60)
    print("R-GCN  —  multi-relational T1a  —  spatial CV (8 blocs)")
    print("=" * 60)
    t0 = time.time()
    res_rgcn = H.run_t1_multirel_cv(
        df, name="rgcn", regime="spatial",
        compute_delta=COMPUTE_DELTA, write=True,
        k_spatial=K_SPATIAL, cap_km_spatial=CAP_KM_SPATIAL,
        k_subbasin=K_SUBBASIN, cap_km_subbasin=CAP_KM_SUBBASIN,
        hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT, heads=HEADS,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, lr=LR,
        weight_decay=WEIGHT_DECAY, n_blocks=N_BLOCKS, seed=SEED,
        exp_dir=EXP_DIR, verbose=True)
    elapsed_rgcn = time.time() - t0

    sp = res_rgcn["spatial"]
    gm = sp["global_metrics"]
    ci = sp["auc_oof_ci95"]
    assert sp["n_cross_block_total"] == 0, "LEAK: cross-block edges remain!"

    print(f"\nR-GCN spatial AUC : {sp['auc_oof_global']:.4f}  "
          f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]")
    print(f"          mean±std: {sp['auc_mean']:.4f} ± {sp['auc_std']:.4f}")
    print(f"          F1@OOF  : {gm['f1']:.4f}   PR-AUC: {gm['pr_auc']:.4f}")
    print(f"          Brier   : {gm['brier']:.4f}   ECE: {gm.get('ece', float('nan')):.4f}")
    if COMPUTE_DELTA and "delta_random_minus_spatial" in res_rgcn:
        print(f"          Δ(rand−spatial): {res_rgcn['delta_random_minus_spatial']:+.4f}")
    print(f"          cross_block_total: {sp['n_cross_block_total']}  (must be 0)")
    print(f"Elapsed: {elapsed_rgcn/60:.1f} min")

    with open(f"{EXP_DIR}/metrics_rgcn.json", "w") as f:
        json.dump(res_rgcn, f, indent=2, default=_cvt)
    print("Checkpoint written to", f"{EXP_DIR}/metrics_rgcn.json")
else:
    res_rgcn = None
    print("SMOKE_TEST=True — skipping full R-GCN run.")

## Cell 8 — Comparison table (HGT vs R-GCN vs baselines)

In [ ]:
import numpy as np

# gnn_phase1 reference values (single-relation, same protocol)
BASELINES = {
    "GraphSAGE (spatial, phase1)": {"auc": 0.618, "auc_std": 0.067, "delta": 0.196},
    "GCN       (spatial, phase1)": {"auc": 0.624, "auc_std": 0.074, "delta": 0.218},
    "RF wall   (spatial, tabular)": {"auc": 0.600, "auc_std": None,  "delta": None},
}

print(f"{'Model':<35} {'AUC spatial':>12} {'AUC 95%CI':>20} {'±std':>8} {'Δrand':>8} {'Brier':>7} {'ECE':>7}")
print("-" * 102)

for label, bv in BASELINES.items():
    std = f"±{bv['auc_std']:.3f}" if bv["auc_std"] else "      "
    delta = f"{bv['delta']:+.3f}" if bv["delta"] else "   — "
    print(f"{label:<35} {bv['auc']:>12.4f}   {'(prior runs)':>20} {std:>8} {delta:>8}")

for model_name, res in [("HGT (spatial, multi-rel)", res_hgt),
                         ("R-GCN (spatial, multi-rel)", res_rgcn)]:
    if res is None:
        print(f"{model_name:<35} {'(not run)':>12}")
        continue
    sp = res["spatial"]; gm = sp["global_metrics"]; ci = sp["auc_oof_ci95"]
    delta = res.get("delta_random_minus_spatial", float("nan"))
    print(f"{model_name:<35} {sp['auc_oof_global']:>12.4f}   "
          f"[{ci['ci_low']:.3f},{ci['ci_high']:.3f}]{'':>5} "
          f"±{sp['auc_std']:.3f}  {delta:>+8.3f} "
          f"{gm['brier']:>7.4f} {gm.get('ece', float('nan')):>7.4f}")

print()
print("NOTE: gains below inter-fold σ (~0.06–0.07) are within noise (eval C-CMP).")
print("Δ(random−spatial) ≈ spatial-leakage inflation; random AUC is NOT a valid score.")
print("Dong et al. (2024) reports high AUC with random split — not comparable here.")

## Cell 9 — Persist results (IMPORTANT: Colab workspace is ephemeral)

> **Warning**: all files in `/content/pfas-gnn/` are lost when the Colab runtime disconnects.
> Run this cell to save your results before closing the session.

In [ ]:
import shutil, os

# Option A: Download results archive
archive = "hgt_rgcn_t1_results"
shutil.make_archive(archive, "zip", EXP_DIR)
print(f"Archive: {archive}.zip ({os.path.getsize(archive+'.zip') // 1024} KB)")
if IN_COLAB:
    from google.colab import files
    files.download(f"{archive}.zip")
    print("Download triggered.")
else:
    print(f"Local run: archive at {os.path.abspath(archive+'.zip')}")

# Option B: Git commit + push (only if remote is authenticated)
print("\n--- Option B: git commit + push ---")
print("Uncomment and run the block below if you have push access:")
# subprocess.run(["git", "add", EXP_DIR], cwd=REPO_DIR)
# subprocess.run(["git", "commit", "-m",
#     f"results(hgt_rgcn_t1): HGT/RGCN spatial AUC from Colab GPU run"],
#     cwd=REPO_DIR)
# subprocess.run(["git", "push"], cwd=REPO_DIR)